In [0]:
# Databricks notebook: 04_optimize_delta
# Optimizes all Delta tables (compaction + Z-order) for schemas 01_bronze, 02_silver, 03_gold

from delta.tables import DeltaTable

catalog_name = "retail_catalog"
schemas = ["01_bronze", "02_silver", "03_gold"]

def zorder_cols(table_name: str):
    name = table_name.lower()
    if "date" in name:
        return ["datekey"]
    if "customer" in name:
        return ["customerid"]
    if "product" in name:
        return ["productid"]
    if "store" in name:
        return ["storeid"]
    if "promo" in name:
        return ["promoid"]
    if "fact" in name or "sales" in name:
        return ["datekey", "productid", "customerid"]
    return None

for schema in schemas:
    full_schema = f"{catalog_name}.{schema}"
    tables = spark.sql(f"SHOW TABLES IN {full_schema}").collect()
    for row in tables:
        table_name = row.tableName
        full_name = f"{full_schema}.{table_name}"
        print(f"Optimizing {full_name}...")
        dt = DeltaTable.forName(spark, full_name)
        zcols = zorder_cols(table_name)
        if zcols:
            existing_cols = spark.table(full_name).columns
            valid = [c for c in zcols if c in existing_cols]
            if valid:
                dt.optimize().executeZOrderBy(valid)
                print(f"  -> Z-ORDER BY {valid} applied")
                continue
        dt.optimize().executeCompaction()
        print("  -> Compaction applied")

print("Optimization completed.")

Optimizing retail_catalog.01_bronze.bronze_dimcustomer...
  -> Z-ORDER BY ['customerid'] applied
Optimizing retail_catalog.01_bronze.bronze_dimdate...
  -> Z-ORDER BY ['datekey'] applied
Optimizing retail_catalog.01_bronze.bronze_dimproduct...
  -> Z-ORDER BY ['productid'] applied
Optimizing retail_catalog.01_bronze.bronze_dimpromotion...
  -> Z-ORDER BY ['promoid'] applied
Optimizing retail_catalog.01_bronze.bronze_dimstore...
  -> Z-ORDER BY ['storeid'] applied
Optimizing retail_catalog.01_bronze.bronze_factsales...
  -> Z-ORDER BY ['datekey', 'productid', 'customerid'] applied
Optimizing retail_catalog.02_silver.silver_dimcustomer...
  -> Z-ORDER BY ['customerid'] applied
Optimizing retail_catalog.02_silver.silver_dimdate...
  -> Z-ORDER BY ['datekey'] applied
Optimizing retail_catalog.02_silver.silver_dimproduct...
  -> Z-ORDER BY ['productid'] applied
Optimizing retail_catalog.02_silver.silver_dimpromotion...
  -> Z-ORDER BY ['promoid'] applied
Optimizing retail_catalog.02_silver.